[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/history/h12_e_history.ipynb)

# A brief history of e

The number nobody invented, to try with your own hands: pay Bernoulli's bank more and more often, build e to the x
out of small steps, measure the area under one over x, shuffle letters into the wrong envelopes thousands of times,
test the 37% rule, and watch carbon-14 run down to one over e.

**How to use this notebook.** The grey boxes are cells. Click the first one, then press **Shift + Enter** (or the
▶ Run button) to run it. Run each cell in turn, from the top to the bottom. When a slider or a box appears, change
it and watch. Nothing can break: if something looks odd, run everything again from the top. It works in your
browser (no install) and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, and a few helpers. Run this cell first.
import math
import random
from fractions import Fraction
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider, ToggleButton

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "figure.dpi": 100,
})

E = math.e
assert abs(sum(1 / math.factorial(k) for k in range(20)) - E) < 1e-15
print(f"Ready. e = {E:.15f}…  Run the next cell.")

## 1 · Bernoulli's bank

The bank pays 100% interest a year, split into equal parts paid during the year. Slide how many times it pays.
Does one coin grow without end?

In [ ]:
def bank(times_a_year=12):
    n = times_a_year
    values = [(1 + 1 / k) ** k for k in range(1, n + 1)]
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(range(1, n + 1), values, color=TEAL, marker="o" if n <= 60 else None, ms=4)
    ax.axhline(E, color=CORAL, lw=2, label=f"e = {E:.6f}…")
    ax.set_xlabel("payments a year"); ax.set_ylabel("coins after a year"); ax.set_ylim(1.9, 2.8); ax.legend()
    ax.set_title("more payments help less and less")
    plt.show()
    print(f"paid {n} times a year: one coin becomes {(1 + 1 / n) ** n:.7f}   (e = {E:.7f}…)")

interact(bank, times_a_year=IntSlider(value=12, min=1, max=1000, description="times a year"));

## 2 · Build e to the x out of small steps

Growth whose speed equals its size: at each step, add the amount times the step. Slide the number of steps in one
year and compare the stairs with the smooth curve e to the x.

In [ ]:
def stairs(steps=10):
    n = steps
    xs, ys, amount = [0.0], [1.0], 1.0
    for k in range(n):
        amount += amount * (1 / n)
        xs += [(k + 1) / n, (k + 1) / n]; ys += [ys[-1], amount]
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(xs, ys, color=TEAL, lw=2, label=f"{n} steps")
    grid = [i / 200 for i in range(201)]
    ax.plot(grid, [math.exp(x) for x in grid], color=CORAL, lw=2, label="e to the x")
    ax.set_xlabel("time (years)"); ax.set_ylabel("amount"); ax.legend()
    plt.show()
    print(f"{n} steps end at {amount:.6f}; e to the 1 is {E:.6f}")

interact(stairs, steps=IntSlider(value=10, min=1, max=500, description="steps"));

## 3 · The area under one over x

Saint-Vincent's curve. Slide the end point and watch the shaded area. Where does the area reach exactly one?

In [ ]:
def area_under(end=2.0, strips=2000):
    h = (end - 1) / strips
    area = sum(1 / (1 + (i + 0.5) * h) for i in range(strips)) * h
    fig, ax = plt.subplots(figsize=(11, 4))
    xs = [0.6 + i * 0.01 for i in range(int((max(end, 4) - 0.6) / 0.01) + 1)]
    ax.plot(xs, [1 / x for x in xs], color=TEAL, lw=2)
    fill = [1 + i * (end - 1) / 200 for i in range(201)]
    ax.fill_between(fill, [1 / x for x in fill], color=CORAL if abs(area - 1) < 0.005 else SAGE, alpha=0.5)
    ax.set_xlabel("x"); ax.set_ylabel("one over x"); ax.set_title(f"area from 1 to {end:.3f} = {area:.4f}")
    plt.show()
    print(f"area from 1 to {end:.3f}: {area:.4f}   (it is exactly 1 at e = {E:.4f}…)")

interact(area_under, end=FloatSlider(value=2.0, min=1.05, max=4.0, step=0.001, description="end point",
                                     readout_format=".3f"));

## 4 · Letters in the wrong envelopes

Put n letters into n addressed envelopes at random, again and again. How often does not one letter reach the
right person? Compare with the exact answer and with one over e.

In [ ]:
def envelopes(letters=10, tries=20000, seed=1708):
    rng = random.Random(seed)
    none_right = 0
    for _ in range(tries):
        order = list(range(letters)); rng.shuffle(order)
        none_right += all(order[i] != i for i in range(letters))
    exact = sum((-1) ** k / math.factorial(k) for k in range(letters + 1))
    print(f"{letters} letters, {tries:,} shuffles: none right {none_right / tries:.4f} of the time")
    print(f"exact: {exact:.6f}    one over e: {1 / E:.6f}")

interact(envelopes, letters=IntSlider(value=10, min=2, max=40, description="letters"),
         tries=IntSlider(value=20000, min=1000, max=100000, step=1000, description="shuffles"),
         seed=IntSlider(value=1708, min=1, max=5000, description="seed"));

## 5 · When to stop looking (the 37% rule)

A hundred slips with different numbers come one at a time. Skip the first few, then take the first that beats them
all. Slide how many you skip and see how often you land the very biggest.

In [ ]:
def stop_rule(skip=37, slips=100):
    def win(r):
        if r == 0: return 1 / slips
        return r / slips * sum(1 / (i - 1) for i in range(r + 1, slips + 1))
    chances = [win(r) for r in range(slips)]
    best = max(range(slips), key=lambda r: chances[r])
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(range(slips), chances, color=TEAL, lw=2)
    ax.axvline(skip, color=CORAL, lw=2, label=f"you skip {skip}")
    ax.axhline(1 / E, color=SAGE, lw=1.5, ls="--", label="one over e")
    ax.set_xlabel("how many you skip"); ax.set_ylabel("chance of the very biggest"); ax.legend()
    plt.show()
    print(f"skip {skip}: you land the biggest {chances[skip]:.3f} of the time; the best plan skips {best} → {chances[best]:.3f}")

interact(stop_rule, skip=IntSlider(value=37, min=0, max=99, description="skip"));

## 6 · Carbon-14 runs down to one over e

Each atom has the same tiny chance of breaking down at any moment. Slide the years and read what share is left.

In [ ]:
HALF = 5730                                  # half-life in years
TAU = HALF / math.log(2)                     # the mean life: about 8,267 years
def carbon(years=8267):
    share = 0.5 ** (years / HALF)
    fig, ax = plt.subplots(figsize=(11, 4))
    ys = list(range(0, 20001, 50))
    ax.plot(ys, [0.5 ** (y / HALF) for y in ys], color=TEAL, lw=2)
    ax.axhline(1 / E, color=CORAL, lw=1.5, ls="--", label="one over e")
    ax.axvline(years, color=SAGE, lw=2)
    ax.set_xlabel("years"); ax.set_ylabel("share left"); ax.legend()
    plt.show()
    print(f"after {years:,} years, {share:.3f} is left   (one over e = {1 / E:.3f} at about {TAU:,.0f} years)")

interact(carbon, years=IntSlider(value=8267, min=0, max=20000, step=50, description="years"));

## 7 · The challenge from the video

Six friends put their names into a hat and each draws one at random. What's the chance that nobody draws their own
name: closer to a half, a third or a quarter? Try it first, then press the button.

In [ ]:
def challenge(show_answer=False):
    if not show_answer:
        print("six names, six people: nobody draws their own?  ½, ⅓ or ¼?   (press the button when you're ready)"); return
    derangements = round(math.factorial(6) * sum((-1) ** k / math.factorial(k) for k in range(7)))
    f = Fraction(derangements, math.factorial(6))
    print(f"{derangements} of the 720 ways leave nobody with their own name: {f} = {float(f):.4f}")
    print(f"closest to a third, and already almost exactly one over e ({1 / E:.4f})")

interact(challenge, show_answer=ToggleButton(value=False, description="show the answer"));

## Sources
John Napier, *Mirifici Logarithmorum Canonis Descriptio* (1614); Grégoire de Saint-Vincent, *Opus geometricum*
(1647), and A. A. de Sarasa (1649); Jakob Bernoulli (1683) and Leibniz's letters to Huygens (1690–91), MacTutor,
*The number e*; Euler, *Mechanica* (1736) and *Introductio* (1748); Montmort, *Essay d'analyse sur les jeux de
hazard* (1708; 1713); M. Gardner, *Scientific American* (1960) and T. Ferguson, *Statistical Science* (1989);
C. Hermite (1873); A. de Moivre (1733); radiocarbon half-life 5,730 years. All retrieved 2026-11-16. Every number
above is computed in the cells.